# Load the Data

In [2]:
import pandas as pd 


In [3]:
xl = pd.read_excel('../data/Customer_Churn_Data_Large.xlsx', sheet_name=None)


In [4]:
print(list(xl.keys()))

['Customer_Demographics', 'Transaction_History', 'Customer_Service', 'Online_Activity', 'Churn_Status']


In [5]:
demographics = xl['Customer_Demographics']
transactions = xl['Transaction_History']
service = xl['Customer_Service']
online = xl['Online_Activity']
churn = xl['Churn_Status']

In [6]:
def overview(df):
    print("Shape of Dataset:", df.shape)
    print("\nData Type\n", df.dtypes)
    print("\nFirst five rows\n", df.head())
    print("\nNumber of missing values\n", df.isnull().sum())
    print("\n Number of unique values\n ",df.nunique())

In [7]:
overview(demographics)

Shape of Dataset: (1000, 5)

Data Type
 CustomerID        int64
Age               int64
Gender           object
MaritalStatus    object
IncomeLevel      object
dtype: object

First five rows
    CustomerID  Age Gender MaritalStatus IncomeLevel
0           1   62      M        Single         Low
1           2   65      M       Married         Low
2           3   18      M        Single         Low
3           4   21      M       Widowed         Low
4           5   21      M      Divorced      Medium

Number of missing values
 CustomerID       0
Age              0
Gender           0
MaritalStatus    0
IncomeLevel      0
dtype: int64

 Number of unique values
  CustomerID       1000
Age                52
Gender              2
MaritalStatus       4
IncomeLevel         3
dtype: int64


In [8]:
overview(transactions)

Shape of Dataset: (5054, 5)

Data Type
 CustomerID                  int64
TransactionID               int64
TransactionDate    datetime64[ns]
AmountSpent               float64
ProductCategory            object
dtype: object

First five rows
    CustomerID  TransactionID TransactionDate  AmountSpent ProductCategory
0           1           7194      2022-03-27       416.50     Electronics
1           2           7250      2022-08-08        54.96        Clothing
2           2           9660      2022-07-25       197.50     Electronics
3           2           2998      2022-01-25       101.31       Furniture
4           2           1228      2022-07-24       397.37        Clothing

Number of missing values
 CustomerID         0
TransactionID      0
TransactionDate    0
AmountSpent        0
ProductCategory    0
dtype: int64

 Number of unique values
  CustomerID         1000
TransactionID      3864
TransactionDate     365
AmountSpent        4797
ProductCategory       5
dtype: int64


In [9]:
overview(service)

Shape of Dataset: (1002, 5)

Data Type
 CustomerID                   int64
InteractionID                int64
InteractionDate     datetime64[ns]
InteractionType             object
ResolutionStatus            object
dtype: object

First five rows
    CustomerID  InteractionID InteractionDate InteractionType ResolutionStatus
0           1           6363      2022-03-31         Inquiry         Resolved
1           2           3329      2022-03-17         Inquiry         Resolved
2           3           9976      2022-08-24         Inquiry         Resolved
3           4           7354      2022-11-18         Inquiry         Resolved
4           4           5393      2022-07-03         Inquiry       Unresolved

Number of missing values
 CustomerID          0
InteractionID       0
InteractionDate     0
InteractionType     0
ResolutionStatus    0
dtype: int64

 Number of unique values
  CustomerID          668
InteractionID       960
InteractionDate     344
InteractionType       3
ResolutionS

In [10]:
overview(online)

Shape of Dataset: (1000, 4)

Data Type
 CustomerID                 int64
LastLoginDate     datetime64[ns]
LoginFrequency             int64
ServiceUsage              object
dtype: object

First five rows
    CustomerID LastLoginDate  LoginFrequency ServiceUsage
0           1    2023-10-21              34   Mobile App
1           2    2023-12-05               5      Website
2           3    2023-11-15               3      Website
3           4    2023-08-25               2      Website
4           5    2023-10-27              41      Website

Number of missing values
 CustomerID        0
LastLoginDate     0
LoginFrequency    0
ServiceUsage      0
dtype: int64

 Number of unique values
  CustomerID        1000
LastLoginDate      340
LoginFrequency      49
ServiceUsage         3
dtype: int64


In [11]:
overview(churn)

Shape of Dataset: (1000, 2)

Data Type
 CustomerID     int64
ChurnStatus    int64
dtype: object

First five rows
    CustomerID  ChurnStatus
0           1            0
1           2            1
2           3            0
3           4            0
4           5            0

Number of missing values
 CustomerID     0
ChurnStatus    0
dtype: int64

 Number of unique values
  CustomerID     1000
ChurnStatus       2
dtype: int64


### We have average of 5 transactions per customer, we must aggregate before merging

In [12]:
# example transaction
transactions.loc[transactions['CustomerID'] == 2]

,CustomerID,TransactionID,TransactionDate,AmountSpent,ProductCategory
1,2,7250,2022-08-08,54.96,Clothing
2,2,9660,2022-07-25,197.50,Electronics
3,2,2998,2022-01-25,101.31,Furniture
4,2,1228,2022-07-24,397.37,Clothing
5,2,8903,2022-01-09,285.21,Electronics
6,2,3527,2022-09-16,311.34,Electronics
7,2,9279,2022-11-19,199.73,Groceries


In [13]:
# Aggregate Transaction_History (5054 to 1000 rows)
transactions['TransactionDate'] = pd.to_datetime(transactions['TransactionDate'])
ref = transactions['TransactionDate'].max()  

transactions_aggregated = transactions.groupby('CustomerID').agg(
        TotalSpend = ('AmountSpent', 'sum'),
        AvgSpend = ('AmountSpent', 'mean'),
        NumTransactions = ('TransactionID', 'count'),
        FavCategory = ('ProductCategory', lambda x: x.mode()[0]),
        DaysSinceLastTx = ('TransactionDate', lambda x: (ref-x.max()).days)
        
    ).reset_index()

transactions_aggregated


,CustomerID,TotalSpend,AvgSpend,NumTransactions,FavCategory,DaysSinceLastTx
0,1,416.50,416.500000,1,Electronics,279
1,2,1547.42,221.060000,7,Electronics,42
2,3,1702.98,283.830000,6,Furniture,84
3,4,917.29,183.458000,5,Electronics,4
4,5,2001.49,250.186250,8,Electronics,10
...,...,...,...,...,...,...
995,996,227.25,227.250000,1,Books,160
996,997,419.82,209.910000,2,Electronics,67
997,998,252.15,252.150000,1,Books,104
998,999,2393.26,265.917778,9,Furniture,24


### Check for missing customer service records

In [14]:
all_customers     = set(demographics['CustomerID'])
service_customers = set(service['CustomerID'])

missing = all_customers - service_customers
print(len(missing))         
print(sorted(missing)[:10])

332
[5, 7, 10, 21, 32, 39, 42, 46, 47, 48]


In [15]:
service.columns

Index(['CustomerID', 'InteractionID', 'InteractionDate', 'InteractionType',
       'ResolutionStatus'],
      dtype='object')

In [16]:
service['InteractionType'].unique(), service['ResolutionStatus'].unique()

(array(['Inquiry', 'Feedback', 'Complaint'], dtype=object),
 array(['Resolved', 'Unresolved'], dtype=object))

In [17]:
# We are missing 332 service records we must aggregate it
service_aggregated = service.groupby('CustomerID').agg(
    NumInteractions = ('InteractionID', 'count'),
    NumComplaints = ('InteractionType', lambda x: (x=='Complaint').sum()),
    NumUnresolved = ('ResolutionStatus', lambda x: (x=='Unresolved').sum())
).reset_index()
service_aggregated

,CustomerID,NumInteractions,NumComplaints,NumUnresolved
0,1,1,0,0
1,2,1,0,0
2,3,1,0,0
3,4,2,0,1
4,6,1,0,0
...,...,...,...,...
663,989,2,2,2
664,990,2,1,1
665,992,1,0,1
666,994,2,2,2


### Now we can combine all sheets

In [19]:
# left join

df = demographics.merge(transactions_aggregated, on='CustomerID', how='left') \
    .merge(service_aggregated, on='CustomerID', how='left') \
        .merge(online[['CustomerID', 'LoginFrequency', 'LastLoginDate', 'ServiceUsage']], on='CustomerID', how='left') \
            .merge(churn, on='CustomerID', how='left')

In [20]:
df.head()

,CustomerID,Age,Gender,MaritalStatus,IncomeLevel,TotalSpend,AvgSpend,NumTransactions,FavCategory,DaysSinceLastTx,NumInteractions,NumComplaints,NumUnresolved,LoginFrequency,LastLoginDate,ServiceUsage,ChurnStatus
0,1,62,M,Single,Low,416.50,416.50000,1,Electronics,279,1.0,0.0,0.0,34,2023-10-21,Mobile App,0
1,2,65,M,Married,Low,1547.42,221.06000,7,Electronics,42,1.0,0.0,0.0,5,2023-12-05,Website,1
2,3,18,M,Single,Low,1702.98,283.83000,6,Furniture,84,1.0,0.0,0.0,3,2023-11-15,Website,0
3,4,21,M,Widowed,Low,917.29,183.45800,5,Electronics,4,2.0,0.0,1.0,2,2023-08-25,Website,0
4,5,21,M,Divorced,Medium,2001.49,250.18625,8,Electronics,10,NaN,NaN,NaN,41,2023-10-27,Website,0


In [22]:
# Fill the NaN for the 332 customers with no service record
for col in ['NumInteractions', 'NumComplaints', 'NumUnresolved']:
    df[col] = df[col].fillna(0)

In [23]:
print(df.shape)
print(df.isnull().any())

(1000, 17)
CustomerID         False
Age                False
Gender             False
MaritalStatus      False
IncomeLevel        False
TotalSpend         False
AvgSpend           False
NumTransactions    False
FavCategory        False
DaysSinceLastTx    False
NumInteractions    False
NumComplaints      False
NumUnresolved      False
LoginFrequency     False
LastLoginDate      False
ServiceUsage       False
ChurnStatus        False
dtype: bool
